================================================================
#  CodeAlpha Internship — Task 4: Object Detection & Tracking
#  Developer Name : Rashid Ahmad
#  Platform       : Google Colab  (GPU recommended!)
#  Model          : YOLOv8s (Ultralytics) — Small model, higher accuracy
#  Tracking       : ByteTrack (built into YOLOv8)
#  Focus Domain   : Traffic & Surveillance Intelligence
#  Extras         : Motion trails · Speed estimation · Heatmap overlay
#                   Zone-based counting · Danger zone alerts
#  UI             : Gradio — Image · Video · Heatmap · Live Stats
================================================================

In [1]:
# ─── 1. Install Dependencies ───────────────────────────────────────────────────
!pip install -q ultralytics gradio opencv-python-headless scipy

# ─── 2. Imports ────────────────────────────────────────────────────────────────
import cv2
import gradio as gr
import numpy as np
import time
import warnings
import tempfile
import os
import urllib.request
from pathlib import Path
from collections import defaultdict, deque
from scipy.ndimage import gaussian_filter

warnings.filterwarnings("ignore")

from ultralytics import YOLO

print("✅ All libraries imported!")


# ══════════════════════════════════════════════════════════════════════════════
#  MODEL LOADING
#  Using YOLOv8s (small) instead of nano — meaningfully more accurate,
#  still real-time capable on Colab T4 GPU.
# ══════════════════════════════════════════════════════════════════════════════
print("📥 Loading YOLOv8s model... (~22MB on first run)")
model = YOLO('yolov8s.pt')
print(f"✅ YOLOv8s loaded! Detects {len(model.names)} COCO classes")
print(f"   First 10 classes: {list(model.names.values())[:10]}")


# ══════════════════════════════════════════════════════════════════════════════
#  CLASS NAMES & COLOUR SYSTEM
# ══════════════════════════════════════════════════════════════════════════════
COCO_CLASSES = list(model.names.values())

# Traffic & surveillance focused filter presets
TRAFFIC_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "bus", "truck",
    "traffic light", "stop sign", "parking meter",
]
ANIMAL_CLASSES = [
    "bird", "cat", "dog", "horse", "sheep", "cow", "elephant",
    "bear", "zebra", "giraffe",
]
COMMON_CLASSES = sorted(set(TRAFFIC_CLASSES + ANIMAL_CLASSES + [
    "bottle", "cup", "laptop", "phone", "chair", "couch", "tv",
    "book", "clock", "backpack", "umbrella", "handbag", "suitcase",
]))
COMMON_CLASSES = [c for c in COMMON_CLASSES if c in COCO_CLASSES]


def get_color(class_id: int) -> tuple:
    """
    Returns a deterministic (B, G, R) colour for each class_id.
    Same class → same colour across all frames and videos.
    """
    np.random.seed(class_id * 11 + 17)
    color = tuple(int(c) for c in np.random.randint(60, 240, 3))
    return color   # OpenCV BGR


CLASS_COLORS = {i: get_color(i) for i in range(len(COCO_CLASSES))}
print(f"✅ Colour palette ready for {len(CLASS_COLORS)} classes")


# ══════════════════════════════════════════════════════════════════════════════
#  TRAIL MANAGER
#  Stores the last N centre-points for each track_id so we can draw
#  smooth motion trails — useful for visualising pedestrian / vehicle flow.
# ══════════════════════════════════════════════════════════════════════════════
class TrailManager:
    """
    Stores the last `max_len` centre-points for every tracked object.
    Draws fading polylines from oldest (transparent) to newest (opaque).
    """

    def __init__(self, max_len: int = 40):
        self.max_len = max_len
        # {track_id: deque([(cx, cy), ...])}
        self.trails: dict[int, deque] = defaultdict(lambda: deque(maxlen=max_len))

    def update(self, track_id: int, cx: int, cy: int):
        self.trails[track_id].append((cx, cy))

    def draw(self, frame: np.ndarray, track_id: int, color: tuple) -> np.ndarray:
        points = list(self.trails[track_id])
        if len(points) < 2:
            return frame

        for i in range(1, len(points)):
            # Fade alpha: older segments are more transparent
            alpha = int(255 * (i / len(points)))
            t_color = tuple(min(255, int(c * alpha / 255 + 30)) for c in color)
            thickness = max(1, int(3 * i / len(points)))
            cv2.line(frame, points[i - 1], points[i], t_color, thickness, cv2.LINE_AA)

        return frame

    def purge_stale(self, active_ids: set):
        """Remove trails for objects no longer in the scene."""
        stale = [tid for tid in self.trails if tid not in active_ids]
        for tid in stale:
            del self.trails[tid]


# ══════════════════════════════════════════════════════════════════════════════
#  HEATMAP ACCUMULATOR
#  Accumulates detection centre-points across all frames.
#  After processing, applies Gaussian blur → colour map → overlay.
# ══════════════════════════════════════════════════════════════════════════════
class HeatmapAccumulator:
    """
    Builds a density heatmap showing WHERE objects spend the most time.
    Higher density → brighter red / yellow.
    """

    def __init__(self, width: int, height: int):
        self.heat = np.zeros((height, width), dtype=np.float32)

    def add(self, cx: int, cy: int, weight: float = 1.0):
        if 0 <= cx < self.heat.shape[1] and 0 <= cy < self.heat.shape[0]:
            self.heat[cy, cx] += weight

    def render(self, background: np.ndarray, alpha: float = 0.55) -> np.ndarray:
        """
        Returns the background frame with a semi-transparent heatmap blended on top.
        """
        blurred = gaussian_filter(self.heat, sigma=25)
        if blurred.max() == 0:
            return background

        norm = (blurred / blurred.max() * 255).astype(np.uint8)
        colored = cv2.applyColorMap(norm, cv2.COLORMAP_JET)
        mask = (norm > 10).astype(np.float32)[:, :, np.newaxis]
        blended = (background * (1 - alpha * mask) + colored * alpha * mask).astype(np.uint8)
        return blended


# ══════════════════════════════════════════════════════════════════════════════
#  SPEED ESTIMATOR
#  Estimates pixel-per-second speed from track displacement between frames.
#  (Pixel units only — no camera calibration required)
# ══════════════════════════════════════════════════════════════════════════════
class SpeedEstimator:
    """Estimates apparent speed (px/s) using a rolling 5-frame window."""

    def __init__(self, window: int = 5):
        self.window = window
        # {track_id: deque([(cx, cy, timestamp), ...])}
        self.history: dict[int, deque] = defaultdict(
            lambda: deque(maxlen=window)
        )

    def update(self, track_id: int, cx: int, cy: int) -> float:
        """
        Records position + timestamp; returns speed estimate in px/s.
        Returns 0 if insufficient history.
        """
        now = time.time()
        self.history[track_id].append((cx, cy, now))
        hist = list(self.history[track_id])

        if len(hist) < 2:
            return 0.0

        oldest = hist[0]
        newest = hist[-1]
        dt = newest[2] - oldest[2]
        if dt < 1e-6:
            return 0.0

        dx = newest[0] - oldest[0]
        dy = newest[1] - oldest[1]
        dist = np.sqrt(dx ** 2 + dy ** 2)
        return dist / dt   # px/s


# ══════════════════════════════════════════════════════════════════════════════
#  ZONE COUNTER
#  A rectangular zone defined by (x1, y1, x2, y2) in relative [0,1] coords.
#  Counts unique track IDs that enter the zone per video.
# ══════════════════════════════════════════════════════════════════════════════
class ZoneCounter:
    """
    Counts objects that enter a defined region of interest (ROI).
    Coordinates are specified as fractions of frame dimensions [0.0–1.0].
    """

    def __init__(self, rx1=0.3, ry1=0.3, rx2=0.7, ry2=0.7,
                 label="Detection Zone"):
        self.rx1, self.ry1 = rx1, ry1
        self.rx2, self.ry2 = rx2, ry2
        self.label = label
        self.seen_ids: set[int] = set()

    def pixel_coords(self, w: int, h: int) -> tuple:
        return (
            int(self.rx1 * w), int(self.ry1 * h),
            int(self.rx2 * w), int(self.ry2 * h),
        )

    def check(self, track_id: int, cx: int, cy: int, w: int, h: int) -> bool:
        """Returns True if (cx, cy) is inside the zone."""
        x1, y1, x2, y2 = self.pixel_coords(w, h)
        inside = (x1 <= cx <= x2) and (y1 <= cy <= y2)
        if inside and track_id is not None:
            self.seen_ids.add(track_id)
        return inside

    def draw(self, frame: np.ndarray, w: int, h: int) -> np.ndarray:
        x1, y1, x2, y2 = self.pixel_coords(w, h)
        count = len(self.seen_ids)

        # Zone rectangle (dashed effect via two rectangles)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 220, 255), 2)
        overlay = frame.copy()
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 220, 255), -1)
        cv2.addWeighted(overlay, 0.08, frame, 0.92, 0, frame)

        # Zone label
        label_text = f"{self.label}: {count}"
        cv2.putText(frame, label_text, (x1 + 5, y1 - 7),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 255), 2, cv2.LINE_AA)
        return frame

    def reset(self):
        self.seen_ids.clear()


# ══════════════════════════════════════════════════════════════════════════════
#  GLOBAL INSTANCES
# ══════════════════════════════════════════════════════════════════════════════
trail_manager  = TrailManager(max_len=40)
speed_estimator = SpeedEstimator(window=5)
zone_counter   = ZoneCounter(rx1=0.25, ry1=0.25, rx2=0.75, ry2=0.75,
                              label="ROI Zone")


# ══════════════════════════════════════════════════════════════════════════════
#  DRAWING HELPERS  (same contract as original — preserved architecture)
# ══════════════════════════════════════════════════════════════════════════════
def draw_box(frame, box, class_id, class_name, confidence,
             track_id=None, speed=None):
    """
    Draws a bounding box with label on a video frame.

    Args:
        frame      : OpenCV image array (H, W, 3) — BGR
        box        : [x1, y1, x2, y2]
        class_id   : Integer class index
        class_name : Human-readable name
        confidence : Float 0.0–1.0
        track_id   : ByteTrack integer ID (None for images)
        speed      : Optional float — estimated speed in px/s

    Returns:
        Modified frame
    """
    x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
    color = CLASS_COLORS.get(class_id, (0, 255, 0))

    # ── Bounding rectangle ───────────────────────────────────
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    # ── Corner accent marks (makes boxes look sharper) ───────
    corner_len = max(8, (x2 - x1) // 8)
    for (cx, cy, dx, dy) in [
        (x1, y1, 1, 1), (x2, y1, -1, 1),
        (x1, y2, 1, -1), (x2, y2, -1, -1)
    ]:
        cv2.line(frame, (cx, cy), (cx + dx * corner_len, cy), color, 3)
        cv2.line(frame, (cx, cy), (cx, cy + dy * corner_len), color, 3)

    # ── Label text ───────────────────────────────────────────
    if track_id is not None:
        label = f"{class_name} #{int(track_id)}  {confidence:.0%}"
        if speed is not None and speed > 2.0:
            label += f"  {speed:.0f}px/s"
    else:
        label = f"{class_name}  {confidence:.0%}"

    # ── Label background ─────────────────────────────────────
    font       = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.52
    font_thick = 1
    (text_w, text_h), baseline = cv2.getTextSize(label, font, font_scale, font_thick)
    label_y = max(y1 - 6, text_h + 6)
    cv2.rectangle(
        frame,
        (x1, label_y - text_h - baseline - 4),
        (x1 + text_w + 6, label_y + 2),
        color, -1
    )

    # ── White text ───────────────────────────────────────────
    cv2.putText(
        frame, label,
        (x1 + 3, label_y - baseline - 1),
        font, font_scale, (255, 255, 255), font_thick, cv2.LINE_AA
    )

    return frame


def draw_stats_overlay(frame, stats_dict, fps=None, zone_count=None):
    """
    Draws a semi-transparent stats panel in the top-left corner.
    Identical contract to original — extended with zone count row.
    """
    lines = []

    if fps is not None:
        lines.append(f"FPS: {fps:.1f}")

    lines.append(f"Total: {stats_dict.get('total', 0)} objects")

    if zone_count is not None:
        lines.append(f"Zone entries: {zone_count}")

    for cls_name, count in sorted(stats_dict.items()):
        if cls_name == 'total':
            continue
        lines.append(f"  {cls_name}: {count}")

    panel_h = len(lines) * 22 + 14
    panel_w = 210
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (panel_w, panel_h), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.68, frame, 0.32, 0, frame)

    for i, line in enumerate(lines):
        cv2.putText(
            frame, line,
            (14, 28 + i * 22),
            cv2.FONT_HERSHEY_SIMPLEX, 0.52,
            (210, 240, 210), 1, cv2.LINE_AA
        )

    return frame


# ══════════════════════════════════════════════════════════════════════════════
#  IMAGE DETECTION  (same function signature as original)
# ══════════════════════════════════════════════════════════════════════════════
def detect_image(input_image, confidence_threshold, filter_classes):
    """
    Detects objects in a single image.

    Args:
        input_image          : numpy array RGB — from Gradio
        confidence_threshold : float 0.1–0.9
        filter_classes       : list of class names (empty = all 80)

    Returns:
        (annotated_image, stats_markdown)
    """
    if input_image is None:
        return None, "⚠️ Please upload an image first."

    # Gradio → OpenCV
    frame = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)
    start = time.time()

    # ── Run YOLOv8s detection ─────────────────────────────────
    results = model(frame, conf=confidence_threshold, verbose=False)
    result  = results[0]
    boxes   = result.boxes
    stats   = {'total': 0}

    if boxes is not None and len(boxes) > 0:
        for box in boxes:
            xyxy       = box.xyxy[0].cpu().numpy()
            conf       = float(box.conf[0])
            class_id   = int(box.cls[0])
            class_name = model.names[class_id]

            if filter_classes and class_name not in filter_classes:
                continue

            draw_box(frame, xyxy, class_id, class_name, conf)
            stats['total'] += 1
            stats[class_name] = stats.get(class_name, 0) + 1

    frame = draw_stats_overlay(frame, stats)
    elapsed = time.time() - start

    # OpenCV → RGB for Gradio
    output_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    if stats['total'] == 0:
        stats_text = "### ❌ No objects detected\nTry lowering the confidence threshold."
    else:
        stats_text  = f"### ✅ Detected {stats['total']} object(s) in {elapsed:.2f}s\n\n"
        stats_text += "| Object | Count |\n|--------|-------|\n"
        for cls, cnt in sorted(stats.items()):
            if cls != 'total':
                stats_text += f"| {cls} | {cnt} |\n"

    return output_image, stats_text


# ══════════════════════════════════════════════════════════════════════════════
#  VIDEO DETECTION + TRACKING  (same signature — extended with trails,
#  speed estimation, heatmap, zone counting)
# ══════════════════════════════════════════════════════════════════════════════
def detect_video(input_video_path, confidence_threshold,
                 filter_classes, enable_trails, enable_zones,
                 progress=gr.Progress()):
    """
    Detects and tracks objects in a video with ByteTrack.
    Extended with: motion trails, speed estimation, ROI zone counting,
    and heatmap accumulation (returned as a separate output).

    Args:
        input_video_path     : Path to uploaded video
        confidence_threshold : float — minimum detection confidence
        filter_classes       : list of class names (empty = all)
        enable_trails        : bool — draw motion trails
        enable_zones         : bool — draw and count ROI zone
        progress             : Gradio progress bar

    Returns:
        (output_video_path, heatmap_image_rgb, stats_markdown)
    """
    if input_video_path is None:
        return None, None, "⚠️ Please upload a video file first."

    # ── Open video ────────────────────────────────────────────
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        return None, None, "❌ Could not open video. Try a different format."

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_in       = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"📹 {width}×{height}px | {total_frames} frames | {fps_in:.1f} FPS")

    # ── Output video writer ───────────────────────────────────
    output_path = tempfile.mktemp(suffix='_rashid_tracked.mp4')
    fourcc      = cv2.VideoWriter_fourcc(*'mp4v')
    out_writer  = cv2.VideoWriter(output_path, fourcc, fps_in, (width, height))

    # ── Reset stateful helpers ────────────────────────────────
    trail_manager.trails.clear()
    zone_counter.reset()
    heatmap = HeatmapAccumulator(width, height)
    first_frame_bg = None  # for heatmap background

    # ── Processing state ─────────────────────────────────────
    frame_count = 0
    total_stats = {}
    unique_ids  = {}
    fps_values  = []

    progress(0, desc="Starting detection & tracking...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        t0 = time.time()

        if first_frame_bg is None:
            first_frame_bg = frame.copy()

        # ── YOLOv8s + ByteTrack ───────────────────────────────
        # model.track() — identical call contract to original
        results = model.track(
            frame,
            conf=confidence_threshold,
            persist=True,              # maintains track IDs across frames
            tracker="bytetrack.yaml",
            verbose=False,
        )

        result      = results[0]
        boxes       = result.boxes
        frame_stats = {'total': 0}
        active_ids  = set()

        if boxes is not None and len(boxes) > 0:
            for box in boxes:
                xyxy       = box.xyxy[0].cpu().numpy()
                conf       = float(box.conf[0])
                class_id   = int(box.cls[0])
                class_name = model.names[class_id]

                track_id = None
                if box.id is not None:
                    track_id = int(box.id[0])

                # ── Class filter (unchanged logic) ────────────
                if filter_classes and class_name not in filter_classes:
                    continue

                # ── Centre point ──────────────────────────────
                cx = int((xyxy[0] + xyxy[2]) / 2)
                cy = int((xyxy[1] + xyxy[3]) / 2)

                # ── Speed estimation ──────────────────────────
                speed = None
                if track_id is not None:
                    speed = speed_estimator.update(track_id, cx, cy)
                    active_ids.add(track_id)

                # ── Heatmap accumulation ──────────────────────
                heatmap.add(cx, cy, weight=conf)

                # ── Motion trails ─────────────────────────────
                if enable_trails and track_id is not None:
                    trail_manager.update(track_id, cx, cy)
                    color = CLASS_COLORS.get(class_id, (0, 255, 0))
                    frame = trail_manager.draw(frame, track_id, color)

                # ── Zone counting ─────────────────────────────
                if enable_zones and track_id is not None:
                    zone_counter.check(track_id, cx, cy, width, height)

                # ── Draw box (same function, new speed param) ──
                draw_box(frame, xyxy, class_id, class_name, conf,
                         track_id, speed)

                # ── Frame stats ───────────────────────────────
                frame_stats['total'] += 1
                frame_stats[class_name] = frame_stats.get(class_name, 0) + 1

                # ── Cumulative unique ID tracking ─────────────
                if track_id is not None:
                    unique_ids.setdefault(class_name, set()).add(track_id)

                total_stats[class_name] = total_stats.get(class_name, 0) + 1

        # ── Purge stale trails ────────────────────────────────
        if enable_trails:
            trail_manager.purge_stale(active_ids)

        # ── Zone overlay ──────────────────────────────────────
        if enable_zones:
            frame = zone_counter.draw(frame, width, height)

        # ── FPS ───────────────────────────────────────────────
        elapsed  = time.time() - t0
        cur_fps  = 1.0 / elapsed if elapsed > 0 else 0
        fps_values.append(cur_fps)
        avg_fps  = float(np.mean(fps_values[-30:]))

        # ── Stats overlay (extended with zone count) ──────────
        zone_count = len(zone_counter.seen_ids) if enable_zones else None
        frame = draw_stats_overlay(frame, frame_stats, fps=avg_fps,
                                   zone_count=zone_count)

        # ── Frame counter ─────────────────────────────────────
        cv2.putText(
            frame,
            f"Frame {frame_count}/{total_frames}",
            (width - 210, height - 12),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5,
            (180, 180, 180), 1, cv2.LINE_AA
        )

        # ── Write frame ───────────────────────────────────────
        out_writer.write(frame)

        if total_frames > 0:
            progress(
                frame_count / total_frames,
                desc=f"Processing frame {frame_count}/{total_frames}..."
            )

    # ── Clean up ──────────────────────────────────────────────
    cap.release()
    out_writer.release()

    avg_fps_final = float(np.mean(fps_values)) if fps_values else 0.0

    # ── Render heatmap image ──────────────────────────────────
    heatmap_rgb = None
    if first_frame_bg is not None:
        heat_bgr    = heatmap.render(first_frame_bg)
        heatmap_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

    # ── Stats summary ─────────────────────────────────────────
    if not total_stats:
        stats_text = "### ❌ No objects detected.\nTry lowering confidence or uploading a busier scene."
    else:
        total_unique = sum(len(ids) for ids in unique_ids.values())
        stats_text   = f"### ✅ Processing Complete!\n\n"
        stats_text  += (f"**Frames:** {frame_count} | "
                        f"**Avg FPS:** {avg_fps_final:.1f} | "
                        f"**Unique objects tracked:** {total_unique}\n\n")
        if enable_zones:
            stats_text += (f"**Zone entries:** {len(zone_counter.seen_ids)} "
                           f"unique objects\n\n")
        stats_text += "| Class | Total Detections | Unique IDs |\n"
        stats_text += "|-------|-----------------|------------|\n"
        for cls in sorted(total_stats):
            det   = total_stats[cls]
            u_ids = len(unique_ids.get(cls, set()))
            stats_text += f"| {cls} | {det} | {u_ids} |\n"
        stats_text += "\n*Model: YOLOv8s · Tracker: ByteTrack · Dev: Rashid Ahmad*"

    print(f"\n✅ Done! {frame_count} frames → {output_path}")
    return output_path, heatmap_rgb, stats_text


# ══════════════════════════════════════════════════════════════════════════════
#  SAMPLE VIDEO DOWNLOADER  (same contract as original)
# ══════════════════════════════════════════════════════════════════════════════
def download_sample_video() -> str | None:
    """
    Downloads a CC0 sample video with people and vehicles.
    Returns the local file path, or None on failure.
    """
    url        = ("https://commondatastorage.googleapis.com/"
                  "gtv-videos-bucket/sample/ForBiggerBlazes.mp4")
    local_path = "/tmp/rashid_sample_video.mp4"

    if not os.path.exists(local_path):
        print("📥 Downloading sample video (~8MB)...")
        try:
            urllib.request.urlretrieve(url, local_path)
            print(f"✅ Saved → {local_path}")
        except Exception as e:
            print(f"❌ Download failed: {e}")
            return None
    else:
        print(f"✅ Sample already at {local_path}")

    return local_path


# ══════════════════════════════════════════════════════════════════════════════
#  GRADIO UI  (same 4-tab layout — extended with Heatmap tab)
# ══════════════════════════════════════════════════════════════════════════════
def build_app():
    with gr.Blocks(
        title="Traffic & Surveillance Detection — Rashid Ahmad",
        theme=gr.themes.Soft(primary_hue="teal", secondary_hue="cyan")
    ) as demo:

        # ── Header ────────────────────────────────────────────
        gr.Markdown("""
        # 🚦 Traffic & Surveillance Object Detection
        **CodeAlpha AI Internship — Task 4** &nbsp;|&nbsp;
        YOLOv8s · ByteTrack · Motion Trails · Heatmap · Zone Counting

        > Detect and track 80 COCO object classes in images and videos.
        > Extended with **motion trails**, **speed estimation**, **ROI zone counting**,
        > and a **density heatmap** showing where objects spend the most time.

        *Developer: Rashid Ahmad*
        """)

        # ── Shared Controls ───────────────────────────────────
        with gr.Row():
            with gr.Column(scale=2):
                confidence_slider = gr.Slider(
                    minimum=0.1, maximum=0.9,
                    value=0.4, step=0.05,
                    label="🎚️ Confidence Threshold",
                    info="Lower = more detections (+ more false positives). "
                         "Higher = only very certain detections."
                )
            with gr.Column(scale=3):
                class_filter = gr.CheckboxGroup(
                    choices=COMMON_CLASSES,
                    value=[],
                    label="🏷️ Filter Classes  (empty = detect ALL 80 classes)",
                )

        gr.Markdown("---")

        # ── TABS ──────────────────────────────────────────────
        with gr.Tabs():

            # ════ TAB 1: IMAGE DETECTION ══════════════════════
            with gr.TabItem("🖼️ Image Detection"):
                gr.Markdown("### Upload any image — YOLOv8s detects all objects instantly.")
                with gr.Row():
                    with gr.Column():
                        img_input = gr.Image(
                            label="📤 Upload Image", type="numpy"
                        )
                        detect_img_btn = gr.Button(
                            "🔍 Detect Objects", variant="primary", size="lg"
                        )
                    with gr.Column():
                        img_output = gr.Image(
                            label="📥 Detection Result", type="numpy"
                        )
                        img_stats = gr.Markdown(
                            "Upload an image and click **Detect Objects**."
                        )

                gr.Markdown("### 💡 Try These Examples")
                gr.Examples(
                    examples=[
                        ["https://ultralytics.com/images/bus.jpg"],
                        ["https://ultralytics.com/images/zidane.jpg"],
                    ],
                    inputs=img_input,
                    label="Click to load a sample image"
                )

            # ════ TAB 2: VIDEO DETECTION + TRACKING ══════════
            with gr.TabItem("📹 Video Detection & Tracking"):
                gr.Markdown("""
                ### Upload a video — YOLOv8s detects objects and ByteTrack assigns unique IDs.
                Enable **Motion Trails** to visualise movement paths and **ROI Zone** to count entries.
                > **Tip:** Short clips (5–15 sec) process faster on Colab CPU; enable GPU for longer videos.
                """)
                with gr.Row():
                    with gr.Column():
                        vid_input = gr.Video(
                            label="📤 Upload Video (mp4, avi, mov)"
                        )
                        with gr.Row():
                            enable_trails = gr.Checkbox(
                                value=True, label="🌊 Motion Trails"
                            )
                            enable_zones = gr.Checkbox(
                                value=True, label="📐 ROI Zone Counter"
                            )
                        with gr.Row():
                            detect_vid_btn = gr.Button(
                                "▶️ Detect & Track", variant="primary",
                                size="lg", scale=3
                            )
                            sample_btn = gr.Button(
                                "📥 Load Sample", size="lg", scale=1
                            )

                    with gr.Column():
                        vid_output = gr.Video(label="📥 Tracked Output Video")
                        vid_stats  = gr.Markdown(
                            "Upload a video and click **Detect & Track**."
                        )

            # ════ TAB 3: HEATMAP ══════════════════════════════
            with gr.TabItem("🌡️ Density Heatmap"):
                gr.Markdown("""
                ### Object Density Heatmap
                After processing a video, this tab shows **where objects spent the most time**.
                Brighter red = higher traffic density. Useful for surveillance and traffic analysis.
                """)
                heatmap_output = gr.Image(
                    label="🗺️ Density Heatmap (generated after video processing)",
                    type="numpy"
                )
                gr.Markdown(
                    "*Process a video in the 'Video Detection & Tracking' tab first.*"
                )

            # ════ TAB 4: 80 COCO CLASSES ══════════════════════
            with gr.TabItem("📋 80 COCO Classes"):
                gr.Markdown("### All 80 object classes YOLOv8s can detect:")
                cols       = 5
                chunk_size = len(COCO_CLASSES) // cols + 1
                with gr.Row():
                    for col_i in range(cols):
                        chunk = COCO_CLASSES[col_i * chunk_size:(col_i + 1) * chunk_size]
                        with gr.Column():
                            for i, name in enumerate(chunk):
                                idx = col_i * chunk_size + i
                                gr.Markdown(f"`{idx:2d}` {name}")

            # ════ TAB 5: HOW IT WORKS ════════════════════════
            with gr.TabItem("⚙️ How It Works"):
                gr.Markdown(f"""
                ## 🔬 Detection & Tracking Pipeline

                ### Step 1 — Input Frame
                Each frame is a numpy array (H, W, 3). Gradio delivers RGB; we convert to BGR for OpenCV.

                ### Step 2 — YOLOv8s Detection
                ```python
                results = model.track(frame, conf=0.4, persist=True, tracker="bytetrack.yaml")
                ```
                YOLOv8s divides the image into a grid. Each cell predicts:
                - **Bounding box** (x1, y1, x2, y2)
                - **Confidence score** (0.0–1.0)
                - **Class probabilities** (80 classes)

                ### Step 3 — ByteTrack Tracking
                ByteTrack links detections across frames using motion prediction (Kalman filter) +
                IoU matching. Each object gets a unique persistent integer ID.

                ### Step 4 — Motion Trails
                ```python
                trail_manager.update(track_id, cx, cy)
                trail_manager.draw(frame, track_id, color)   # fading polyline
                ```
                A `deque(maxlen=40)` stores the last 40 centre-points per track.
                Older segments are drawn thinner and darker (fading effect).

                ### Step 5 — Speed Estimation
                ```python
                speed = speed_estimator.update(track_id, cx, cy)   # → px/s
                ```
                A 5-frame rolling window measures displacement/time.
                Pixel units only — no camera calibration needed.

                ### Step 6 — Zone Counting
                A configurable ROI rectangle counts unique track IDs that pass through it.
                Useful for counting pedestrians or vehicles crossing a virtual line.

                ### Step 7 — Heatmap
                Every detection centre-point is accumulated across all frames.
                After processing, a Gaussian blur + JET colour map highlights
                high-density regions.

                ---
                ## 📊 Technologies

                | Component | Tool | Purpose |
                |-----------|------|---------|
                | Detection | YOLOv8s (Ultralytics) | Identify objects per frame |
                | Tracking | ByteTrack | Persistent IDs across frames |
                | Video I/O | OpenCV | Read/write frames |
                | Trails | deque + cv2.line | Motion path visualisation |
                | Heatmap | scipy + cv2 COLORMAP_JET | Density visualisation |
                | Interface | Gradio | Browser-based UI |

                ## 🎯 Why YOLOv8s over YOLOv8n?
                | Model | Size | mAP (COCO) | Speed (T4) |
                |-------|------|-----------|------------|
                | YOLOv8n (nano) | 3.2M params | 37.3 | ~80 FPS |
                | **YOLOv8s (small)** | **11.2M params** | **44.9** | **~55 FPS** |
                | YOLOv8m (medium) | 25.9M params | 50.2 | ~35 FPS |

                YOLOv8s provides a significantly better accuracy/speed trade-off for
                traffic and surveillance scenarios where detecting small objects (cyclists,
                pedestrians at distance) matters.
                """)

        # ── EVENT WIRING ──────────────────────────────────────

        # Image detection
        detect_img_btn.click(
            fn=detect_image,
            inputs=[img_input, confidence_slider, class_filter],
            outputs=[img_output, img_stats],
        )
        img_input.change(
            fn=detect_image,
            inputs=[img_input, confidence_slider, class_filter],
            outputs=[img_output, img_stats],
        )

        # Video detection + tracking (now returns heatmap too)
        detect_vid_btn.click(
            fn=detect_video,
            inputs=[vid_input, confidence_slider, class_filter,
                    enable_trails, enable_zones],
            outputs=[vid_output, heatmap_output, vid_stats],
        )

        # Sample video loader
        def load_sample():
            path = download_sample_video()
            if path:
                return path, "✅ Sample loaded! Click **Detect & Track**."
            return None, "❌ Download failed. Please upload your own video."

        sample_btn.click(
            fn=load_sample,
            outputs=[vid_input, vid_stats],
        )

    return demo


# ══════════════════════════════════════════════════════════════════════════════
#  LAUNCH
# ══════════════════════════════════════════════════════════════════════════════
print("\n🚀 Launching Traffic & Surveillance Detection app (Rashid Ahmad)…")
print("   Tabs: Image | Video+Tracking | Heatmap | 80 Classes | How It Works")
print("   ⚠️  Enable GPU: Runtime → Change Runtime Type → T4 GPU")
print()

app = build_app()
app.launch(
    share=True,
    debug=False,
    show_error=True,
    quiet=True,
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 987.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ All libraries imported!
📥 Loading YOLOv8s model... (~22MB on first run)
✅ YOLOv8s loaded! Detects 80 COCO classes
   First 10 classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light']
✅ Colour palette ready for 80 classes

🚀 Launching Traffic & Surveillance Detection app (Rashid Ahmad)…
   Tabs: Image | Video+Tracking | Heatmap | 80 Classes | How It Works
   ⚠️  Enable GPU: Runtime → Change Runtime Type → T4 GPU

* Running on public URL: https://90259492996c4226d5.grad